# Gemma 3 1B IT - GSM8K Raw Layer Data (v3 lean, run-ready)

Self-contained Colab collector for deterministic GSM8K chain-of-thought
rollouts. It saves raw causal decoder outputs for layers 1-26 as float16 with
shape `[26, generated_tokens, 1152]`. Layer 26 is captured by a forward
pre-hook immediately before the model-level final RMSNorm.

Persistence is deliberately lightweight: arrays are validated in memory,
NPZ is atomically finalized before JSON, JSON is the commit marker, and
resume checks use filename intersections only.



In [1]:
!pip -q install transformers==4.51.3 datasets==3.5.0 accelerate==1.6.0 huggingface_hub==0.30.2 sentencepiece==0.2.0 tqdm==4.67.1



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 147.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 123.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.
diffusers 0.3

## Mount Google Drive and authenticate



In [2]:
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    notebook_login()



## Configuration

Leave `RUN_MODE = "smoke"` for the one-rollout check. A 10,000-rollout run
requires both `RUN_MODE = "full"` and the explicit opt-in
`ALLOW_FULL_RUN = True`. With the default `FORCE_SMOKE_REGENERATE = True`,
smoke mode removes its existing JSON commit marker before collection and
regenerates the smoke artifact pair.



In [3]:
import gc
import json
import os
import platform
import random
import re
import time
import traceback
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path

import accelerate
import datasets as datasets_lib
import huggingface_hub
import numpy as np
import torch
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_grad_enabled(False)

RUN_MODE = "full"
ALLOW_FULL_RUN = True
FORCE_SMOKE_REGENERATE = True

MODES = {
    "smoke": {
        "num_prompts": 1,
        "rollouts_per_prompt": 1,
        "max_new_tokens": 128,
    },
    "pilot": {
        "num_prompts": 2,
        "rollouts_per_prompt": 10,
        "max_new_tokens": 8192,
    },
    "full": {
        "num_prompts": 1000,
        "rollouts_per_prompt": 10,
        "max_new_tokens": 8192,
    },
}
if RUN_MODE not in MODES:
    raise ValueError(f"RUN_MODE must be one of {sorted(MODES)}")
if RUN_MODE == "full" and not ALLOW_FULL_RUN:
    raise RuntimeError(
        "The 10,000-rollout run requires ALLOW_FULL_RUN = True."
    )

MODE = MODES[RUN_MODE]
NUM_PROMPTS = MODE["num_prompts"]
ROLLOUTS_PER_PROMPT = MODE["rollouts_per_prompt"]
MAX_NEW_TOKENS = MODE["max_new_tokens"]
SAMPLE_POOL_SIZE = 1000

MODEL_ID = "google/gemma-3-1b-it"
DATASET_NAME = "openai/gsm8k"
DATASET_CONFIG = "main"
DATASET_SPLIT = "test"

SAMPLE_SEED = 42
ROLLOUT_MASTER_SEED = 42
ROLLOUT_SEED_SCHEME = (
    "numpy.SeedSequence([master_seed, sample_index, rollout_id])"
)

TEMPERATURE = 0.6
TOP_P = 0.95
TOP_K = 64

# Change this path to choose a different Drive destination.
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/gemma3_1b_it_gsm8k_raw26_v3_lean"
)
ROOT = OUTPUT_ROOT / RUN_MODE
RECORDS_DIR = ROOT / "records"
HIDDEN_DIR = ROOT / "raw_layer_outputs"
ERRORS_DIR = ROOT / "errors"
RECORDS_JSONL = ROOT / "records.jsonl"
MANIFEST_PATH = ROOT / "experiment_manifest.json"
SAMPLE_CACHE_DIR = OUTPUT_ROOT / "fixed_samples"

for path in (
    ROOT,
    RECORDS_DIR,
    HIDDEN_DIR,
    ERRORS_DIR,
    SAMPLE_CACHE_DIR,
):
    path.mkdir(parents=True, exist_ok=True)


def atomic_json(path, payload):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp, path)


def atomic_jsonl(path, rows):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp, path)


print("Run mode:", RUN_MODE)
print("Expected rollouts:", NUM_PROMPTS * ROLLOUTS_PER_PROMPT)
print("Output root:", ROOT)



Run mode: full
Expected rollouts: 10000
Output root: /content/drive/MyDrive/gemma3_1b_it_gsm8k_raw26_v3_lean/full


## Fixed GSM8K sample and prompt

The same deterministic 1,000-example sample pool is used by smoke and full
modes, so the smoke rollout is the first rollout of the eventual full run.



In [4]:
gsm8k = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=DATASET_SPLIT,
)
DATASET_FINGERPRINT = str(gsm8k._fingerprint)

SAMPLE_FILE = SAMPLE_CACHE_DIR / (
    f"gsm8k_{DATASET_CONFIG}_{DATASET_SPLIT}"
    f"_seed{SAMPLE_SEED}_n{SAMPLE_POOL_SIZE}"
    f"_{DATASET_FINGERPRINT[:12]}.jsonl"
)

if SAMPLE_FILE.exists():
    with open(SAMPLE_FILE, "r", encoding="utf-8") as handle:
        sample_pool = [
            json.loads(line)
            for line in handle
            if line.strip()
        ]

    if len(sample_pool) != SAMPLE_POOL_SIZE:
        raise RuntimeError(
            f"Expected {SAMPLE_POOL_SIZE} cached samples, "
            f"found {len(sample_pool)}."
        )
    source_indices = [row["source_index"] for row in sample_pool]
    if len(source_indices) != len(set(source_indices)):
        raise RuntimeError("Cached sample source indices are not unique.")
    if any(
        row.get("sample_seed") != SAMPLE_SEED
        or row.get("dataset_fingerprint") != DATASET_FINGERPRINT
        or row.get("source_split") != DATASET_SPLIT
        for row in sample_pool
    ):
        raise RuntimeError("Cached sample metadata does not match this run.")
else:
    rng = np.random.default_rng(SAMPLE_SEED)
    source_indices = rng.choice(
        len(gsm8k),
        size=SAMPLE_POOL_SIZE,
        replace=False,
    )

    sample_pool = []
    for sample_index, source_index in enumerate(source_indices):
        row = gsm8k[int(source_index)]
        sample_pool.append({
            "sample_id": f"s{sample_index:04d}",
            "source_split": DATASET_SPLIT,
            "source_index": int(source_index),
            "sample_seed": SAMPLE_SEED,
            "dataset_fingerprint": DATASET_FINGERPRINT,
            "question": row["question"],
            "reference_solution": row["answer"],
        })

    atomic_jsonl(SAMPLE_FILE, sample_pool)

samples = sample_pool[:NUM_PROMPTS]

PROMPT_TEMPLATE = (
    "Question: {question}\n\n"
    "Please reason step by step. Put your final numeric answer "
    "in the form \\boxed{...}.\n\n"
    "Solution:"
)


def build_prompt(question):
    content = PROMPT_TEMPLATE.replace(
        "{question}",
        str(question),
        1,
    )
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=False,
        add_generation_prompt=True,
    )


print("Fixed sample pool:", SAMPLE_FILE)
print("Prompts in this run:", len(samples))



Fixed sample pool: /content/drive/MyDrive/gemma3_1b_it_gsm8k_raw26_v3_lean/fixed_samples/gsm8k_main_test_seed42_n1000_59ec1b7f9357.jsonl
Prompts in this run: 1000


## v3 box selection, answer evaluation, and original-token alignment

The final recognized marker wins even when it is incomplete or truncated.
Canonical `phase_ranges` are created only for a complete box whose leading
marker boundary is verified against retained original generated token IDs.



In [5]:
PHASES = ("early", "middle", "late")
NUMBER_RE = re.compile(r"[-+]?(?:\d[\d,]*\.?\d*|\.\d+)")
BOX_MARKERS = (r"\boxed", r"\fbox")
INCOMPLETE_BOX_STATUSES = {
    "partial_marker",
    "missing_open_brace",
    "unbalanced_braces",
}


def _complete_or_incomplete_marker(text, marker_start, marker):
    cursor = marker_start + len(marker)
    while cursor < len(text) and text[cursor].isspace():
        cursor += 1

    if cursor >= len(text):
        return {
            "marker": marker,
            "marker_char_start": marker_start,
            "marker_char_end": marker_start + len(marker),
            "status": "missing_open_brace",
            "content": None,
            "box_char_end": None,
        }
    if text[cursor] != "{":
        return None

    depth = 0
    for char_index in range(cursor, len(text)):
        if text[char_index] == "{":
            depth += 1
        elif text[char_index] == "}":
            depth -= 1
            if depth == 0:
                return {
                    "marker": marker,
                    "marker_char_start": marker_start,
                    "marker_char_end": marker_start + len(marker),
                    "status": "complete",
                    "content": text[cursor + 1:char_index].strip(),
                    "box_char_end": char_index + 1,
                }

    return {
        "marker": marker,
        "marker_char_start": marker_start,
        "marker_char_end": marker_start + len(marker),
        "status": "unbalanced_braces",
        "content": None,
        "box_char_end": None,
    }


def boxed_candidates(text, include_partial_trailing_marker=False):
    text = text or ""
    candidates = []

    for marker in BOX_MARKERS:
        position = 0
        while True:
            marker_start = text.find(marker, position)
            if marker_start < 0:
                break
            candidate = _complete_or_incomplete_marker(
                text,
                marker_start,
                marker,
            )
            if candidate is not None:
                candidates.append(candidate)
            position = marker_start + len(marker)

    if include_partial_trailing_marker:
        stripped = text.rstrip()
        for marker in BOX_MARKERS:
            for prefix_length in range(2, len(marker)):
                prefix = marker[:prefix_length]
                if stripped.endswith(prefix):
                    candidates.append({
                        "marker": marker,
                        "marker_prefix": prefix,
                        "marker_char_start": len(stripped) - len(prefix),
                        "marker_char_end": len(stripped),
                        "status": "partial_marker",
                        "content": None,
                        "box_char_end": None,
                    })

    candidates.sort(
        key=lambda item: (
            int(item["marker_char_start"]),
            int(item["marker_char_end"]),
        )
    )
    return candidates


def select_box_candidate(text, was_truncated):
    candidates = boxed_candidates(
        text,
        include_partial_trailing_marker=was_truncated,
    )
    return candidates[-1] if candidates else None


def normalize_numeric(value):
    if value is None:
        return None
    matches = NUMBER_RE.findall(str(value))
    if not matches:
        return None
    cleaned = (
        matches[-1]
        .replace(",", "")
        .replace("$", "")
        .rstrip(".")
    )
    try:
        number = Decimal(cleaned)
    except InvalidOperation:
        return None
    if number == 0:
        return "0"
    rendered = format(number, "f")
    return (
        rendered.rstrip("0").rstrip(".")
        if "." in rendered
        else rendered
    )


def reference_answer(reference_solution):
    raw = (
        reference_solution.rsplit("####", 1)[1]
        if "####" in reference_solution
        else reference_solution
    )
    return normalize_numeric(raw)


def evaluate_answer(
    generated_text,
    reference_solution,
    selected_box,
    was_truncated,
):
    reference = reference_answer(reference_solution)

    if (
        selected_box is not None
        and selected_box.get("status") == "complete"
    ):
        predicted = normalize_numeric(selected_box.get("content"))
        method = "last_valid_boxed"
    elif selected_box is not None:
        return (
            None,
            reference,
            "incomplete_or_truncated_box",
            None,
        )
    elif was_truncated:
        return (
            None,
            reference,
            "truncated_without_complete_box",
            None,
        )
    else:
        predicted = normalize_numeric(generated_text)
        method = (
            "last_number_fallback"
            if predicted is not None
            else "none"
        )

    correct = (
        None
        if predicted is None or reference is None
        else Decimal(predicted) == Decimal(reference)
    )
    return predicted, reference, method, correct


def _decode_ids(token_ids):
    return tokenizer.decode(
        [int(token_id) for token_id in token_ids],
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )


def align_marker_to_original_tokens(
    parse_text,
    original_parse_ids,
    marker_char_start,
):
    if marker_char_start is None:
        return None, "no_marker"

    marker_char_start = int(marker_char_start)
    if marker_char_start < 0 or marker_char_start > len(parse_text):
        return None, "marker_out_of_bounds"

    original_ids = [
        int(token_id)
        for token_id in original_parse_ids
    ]
    if _decode_ids(original_ids) != parse_text:
        return None, "decoded_original_ids_mismatch"

    candidate_indices = set()
    try:
        encoded = tokenizer(
            parse_text,
            add_special_tokens=False,
            return_offsets_mapping=True,
        )
        retokenized_ids = [
            int(token_id)
            for token_id in encoded["input_ids"]
        ]
        if retokenized_ids == original_ids:
            for token_index, (start, end) in enumerate(
                encoded["offset_mapping"]
            ):
                if (
                    int(start)
                    <= marker_char_start
                    < int(end)
                ):
                    candidate_indices.add(int(token_index))
    except Exception:
        pass

    low, high = 0, len(original_ids)
    decoded_cache = {0: ""}

    def decoded_prefix(index):
        if index not in decoded_cache:
            decoded_cache[index] = _decode_ids(
                original_ids[:index]
            )
        return decoded_cache[index]

    while low < high:
        middle = (low + high) // 2
        if len(decoded_prefix(middle)) <= marker_char_start:
            low = middle + 1
        else:
            high = middle

    candidate_indices.add(max(0, low - 1))
    for token_index in range(
        max(0, low - 3),
        min(len(original_ids), low + 2),
    ):
        candidate_indices.add(token_index)

    for token_index in sorted(candidate_indices):
        if token_index >= len(original_ids):
            continue
        before = decoded_prefix(token_index)
        after = decoded_prefix(token_index + 1)
        before_length = len(before)
        after_length = len(after)
        if (
            before == parse_text[:before_length]
            and after == parse_text[:after_length]
            and before_length
            <= marker_char_start
            < after_length
        ):
            return (
                token_index,
                "verified_original_token_boundary",
            )

    return None, "not_original_token_boundary"


def split_phases(token_count):
    if token_count is None:
        return None
    token_count = int(token_count)
    if token_count < 0:
        raise ValueError("token_count cannot be negative")

    quotient, remainder = divmod(token_count, 3)
    lengths = [
        quotient + (1 if index < remainder else 0)
        for index in range(3)
    ]
    ranges = {}
    cursor = 0
    for phase, length in zip(PHASES, lengths):
        ranges[phase] = [cursor, cursor + length]
        cursor += length
    if cursor != token_count:
        raise AssertionError("Phase split failed to cover its input.")
    return ranges


assert select_box_candidate(
    r"draft \boxed{1}; final \fbox{17}",
    False,
)["content"] == "17"
assert select_box_candidate(
    r"draft \boxed{1}; final \boxed{",
    True,
)["status"] == "unbalanced_braces"
assert select_box_candidate(
    "truncated marker \\bo",
    True,
)["status"] == "partial_marker"
assert evaluate_answer(
    r"\boxed{",
    "#### 1",
    select_box_candidate(r"\boxed{", True),
    True,
)[3] is None

print("Parser and phase helper assertions passed.")



Parser and phase helper assertions passed.


## Load Gemma and locate the raw final decoder output



In [6]:
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab.")

MODEL_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if not hasattr(model, "model"):
    raise RuntimeError("Loaded model has no model.model backbone.")
backbone = model.model
backbone.eval()
if not hasattr(backbone, "norm"):
    raise RuntimeError(
        "Gemma backbone has no model-level final RMSNorm."
    )

text_config = getattr(model.config, "text_config", model.config)
NUM_LAYERS = int(text_config.num_hidden_layers)
HIDDEN_SIZE = int(text_config.hidden_size)
if (NUM_LAYERS, HIDDEN_SIZE) != (26, 1152):
    raise RuntimeError(
        "Expected Gemma 3 1B text shape (26, 1152), "
        f"found ({NUM_LAYERS}, {HIDDEN_SIZE})."
    )

MODEL_DEVICE = model.get_input_embeddings().weight.device
BACKBONE_DEVICE = next(backbone.parameters()).device

END_OF_TURN_ID = tokenizer.convert_tokens_to_ids(
    "<end_of_turn>"
)
if (
    END_OF_TURN_ID is None
    or END_OF_TURN_ID == tokenizer.unk_token_id
):
    raise RuntimeError(
        "Tokenizer has no valid <end_of_turn> token ID."
    )

configured_eos_ids = model.generation_config.eos_token_id
if configured_eos_ids is None:
    configured_eos_ids = []
elif isinstance(configured_eos_ids, int):
    configured_eos_ids = [configured_eos_ids]
else:
    configured_eos_ids = list(configured_eos_ids)

STOP_TOKEN_IDS = sorted(set(
    int(token_id)
    for token_id in [
        tokenizer.eos_token_id,
        END_OF_TURN_ID,
        *configured_eos_ids,
    ]
    if token_id is not None
))
ORDINARY_EOS_TOKEN_IDS = (
    set(STOP_TOKEN_IDS) - {END_OF_TURN_ID}
)


MODEL_REVISION = getattr(model.config, "_commit_hash", None)
TOKENIZER_REVISION = (
    getattr(tokenizer, "_commit_hash", None)
    or tokenizer.init_kwargs.get("_commit_hash")
)

experiment_manifest = {
    "dataset": {
        "name": DATASET_NAME,
        "config": DATASET_CONFIG,
        "split": DATASET_SPLIT,
        "fingerprint": DATASET_FINGERPRINT,
        "sample_seed": SAMPLE_SEED,
        "sample_pool_size": SAMPLE_POOL_SIZE,
        "sample_file": str(SAMPLE_FILE),
    },
    "prompt": {
        "template": PROMPT_TEMPLATE,
        "rendering": (
            "tokenizer.apply_chat_template with "
            "add_generation_prompt=True"
        ),
    },
    "seeding": {
        "rollout_master_seed": ROLLOUT_MASTER_SEED,
        "rollout_seed_scheme": ROLLOUT_SEED_SCHEME,
        "rollout_seed_formula": (
            "int(np.random.SeedSequence([master_seed, sample_index, "
            "rollout_id]).generate_state(1, dtype=np.uint32)[0])"
        ),
    },
    "generation": {
        "run_mode": RUN_MODE,
        "num_prompts": NUM_PROMPTS,
        "rollouts_per_prompt": ROLLOUTS_PER_PROMPT,
        "do_sample": True,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "top_k": TOP_K,
        "num_beams": 1,
        "max_new_tokens": MAX_NEW_TOKENS,
        "use_cache": True,
        "stop_token_ids": STOP_TOKEN_IDS,
        "end_of_turn_id": END_OF_TURN_ID,
        "ordinary_eos_token_ids": sorted(
            ORDINARY_EOS_TOKEN_IDS
        ),
    },
    "model": {
        "id": MODEL_ID,
        "resolved_revision": MODEL_REVISION,
        "class": type(model).__name__,
        "dtype": str(next(model.parameters()).dtype),
    },
    "tokenizer": {
        "id": MODEL_ID,
        "resolved_revision": TOKENIZER_REVISION,
        "class": type(tokenizer).__name__,
    },
    "packages": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets_lib.__version__,
        "accelerate": accelerate.__version__,
        "huggingface_hub": huggingface_hub.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
    },
    "hidden_states": {
        "shape": "[26, generated_tokens, 1152]",
        "dtype": "float16",
        "causal_slice": "[P - 1 : P + T - 1]",
        "semantics": (
            "row t is the causal state used to predict "
            "generated token t"
        ),
        "layers_1_to_25": "raw decoder block outputs",
        "layer_26": (
            "raw decoder block output captured by a pre-hook "
            "on backbone.norm"
        ),
        "final_rmsnorm_output_saved": False,
    },
}

if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, "r", encoding="utf-8") as handle:
        existing_manifest = json.load(handle)
    if existing_manifest != experiment_manifest:
        differing_sections = sorted(
            key
            for key in set(existing_manifest) | set(experiment_manifest)
            if existing_manifest.get(key) != experiment_manifest.get(key)
        )
        raise RuntimeError(
            "Existing experiment_manifest.json does not match "
            f"this run. Differing sections: {differing_sections}"
        )
else:
    atomic_json(MANIFEST_PATH, experiment_manifest)

print(
    f"Loaded {MODEL_ID}: "
    f"{NUM_LAYERS} layers, hidden size {HIDDEN_SIZE}"
)
print("Stop token IDs:", STOP_TOKEN_IDS)



Loaded google/gemma-3-1b-it: 26 layers, hidden size 1152
Stop token IDs: [1, 106]


## Deterministic generation and raw causal replay

For prompt length `P` and generated length `T`, every layer uses the causal
slice `[P - 1 : P + T - 1]`; row `t` is the state used to predict generated
token `t`. Layers 1-25 come from the backbone hidden-state tuple. Layer 26
comes from the input to `backbone.norm`, captured by a pre-hook.



In [7]:
def rollout_seed(sample_id, rollout_id):
    sample_index = int(str(sample_id).removeprefix("s"))
    sequence = np.random.SeedSequence([
        ROLLOUT_MASTER_SEED,
        sample_index,
        int(rollout_id),
    ])
    return int(
        sequence.generate_state(
            1,
            dtype=np.uint32,
        )[0]
    )


def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.inference_mode()
def generate_one(sample, rollout_id):
    seed = rollout_seed(sample["sample_id"], rollout_id)
    set_seeds(seed)

    rendered_prompt = build_prompt(sample["question"])
    encoded = tokenizer(
        rendered_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )
    prompt_input_ids = (
        encoded.input_ids[0]
        .detach()
        .cpu()
        .clone()
    )
    prompt_attention_mask = (
        encoded.attention_mask[0]
        .detach()
        .cpu()
        .clone()
    )
    if prompt_attention_mask.shape != prompt_input_ids.shape:
        raise RuntimeError("Prompt attention mask shape mismatch.")

    model_inputs = {
        "input_ids": prompt_input_ids.unsqueeze(0).to(
            MODEL_DEVICE
        ),
        "attention_mask": prompt_attention_mask.unsqueeze(0).to(
            MODEL_DEVICE
        ),
    }

    torch.cuda.synchronize()
    started = time.perf_counter()
    output_ids = model.generate(
        **model_inputs,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        num_beams=1,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=STOP_TOKEN_IDS,
        use_cache=True,
    )[0].detach().cpu()
    torch.cuda.synchronize()
    generation_seconds = time.perf_counter() - started

    prompt_length = len(prompt_input_ids)
    if not torch.equal(
        output_ids[:prompt_length],
        prompt_input_ids,
    ):
        raise RuntimeError(
            "Generated sequence does not preserve prompt IDs."
        )

    generated_ids = output_ids[prompt_length:].clone()
    if len(generated_ids) == 0:
        raise RuntimeError("Generation produced zero tokens.")

    original_parse_ids = [
        int(token_id)
        for token_id in generated_ids.tolist()
    ]
    trailing_special_ids = set(STOP_TOKEN_IDS)
    if tokenizer.pad_token_id is not None:
        trailing_special_ids.add(int(tokenizer.pad_token_id))

    trailing_special_token_count = 0
    while (
        original_parse_ids
        and original_parse_ids[-1] in trailing_special_ids
    ):
        original_parse_ids.pop()
        trailing_special_token_count += 1

    generated_text = _decode_ids(original_parse_ids)
    final_token_id = int(generated_ids[-1])
    hit_max_new_tokens = len(generated_ids) >= MAX_NEW_TOKENS
    stop_reason = (
        "end_of_turn"
        if final_token_id == END_OF_TURN_ID
        else "eos"
        if final_token_id in ORDINARY_EOS_TOKEN_IDS
        else "max_new_tokens"
        if hit_max_new_tokens
        else "other"
    )

    del output_ids
    del model_inputs
    del encoded

    return {
        "seed": seed,
        "rendered_prompt": rendered_prompt,
        "prompt_input_ids": prompt_input_ids,
        "generated_ids": generated_ids,
        "original_parse_ids": original_parse_ids,
        "generated_text": generated_text,
        "final_token_id": final_token_id,
        "hit_max_new_tokens": hit_max_new_tokens,
        "was_truncated": stop_reason == "max_new_tokens",
        "stop_reason": stop_reason,
        "trailing_special_token_count": (
            trailing_special_token_count
        ),
        "generation_seconds": generation_seconds,
    }


@torch.inference_mode()
def replay_raw_causal_layers(
    prompt_input_ids,
    generated_ids,
):
    prompt_input_ids = prompt_input_ids.detach().cpu()
    generated_ids = generated_ids.detach().cpu()
    prompt_length = len(prompt_input_ids)
    generated_length = len(generated_ids)
    if prompt_length < 1 or generated_length < 1:
        raise ValueError(
            "Replay requires non-empty prompt and generation."
        )

    full_input_ids = torch.cat(
        [prompt_input_ids, generated_ids],
        dim=0,
    ).unsqueeze(0).to(BACKBONE_DEVICE)
    attention_mask = torch.ones_like(full_input_ids)

    captured = {
        "calls": 0,
        "raw_layer26": None,
    }

    def capture_pre_final_norm(_module, inputs):
        captured["calls"] += 1
        if not inputs:
            raise RuntimeError(
                "Final RMSNorm hook received no input."
            )
        captured["raw_layer26"] = (
            inputs[0][
                0,
                prompt_length - 1:
                prompt_length + generated_length - 1,
                :,
            ]
            .detach()
            .clone()
        )

    hook = backbone.norm.register_forward_pre_hook(
        capture_pre_final_norm
    )
    torch.cuda.synchronize()
    started = time.perf_counter()
    try:
        outputs = backbone(
            input_ids=full_input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )
        torch.cuda.synchronize()
    finally:
        hook.remove()
    replay_seconds = time.perf_counter() - started

    if captured["calls"] != 1:
        raise RuntimeError(
            "Final RMSNorm pre-hook did not run exactly once."
        )
    if outputs.hidden_states is None:
        raise RuntimeError(
            "Backbone did not return hidden states."
        )
    if len(outputs.hidden_states) != NUM_LAYERS + 1:
        raise RuntimeError(
            "Unexpected hidden-state tuple length: "
            f"{len(outputs.hidden_states)}."
        )

    raw_layer26 = captured["raw_layer26"]
    if raw_layer26 is None:
        raise RuntimeError(
            "Final RMSNorm pre-hook captured no tensor."
        )

    post_norm_final = outputs.hidden_states[-1][
        0,
        prompt_length - 1:
        prompt_length + generated_length - 1,
        :,
    ]
    reconstructed_post_norm = backbone.norm(
        raw_layer26.unsqueeze(0)
    )[0]
    tolerance = (
        2e-2
        if MODEL_DTYPE == torch.bfloat16
        else 2e-3
    )
    torch.testing.assert_close(
        reconstructed_post_norm,
        post_norm_final,
        rtol=tolerance,
        atol=tolerance,
    )
    max_abs_error = float(
        (
            reconstructed_post_norm.float()
            - post_norm_final.float()
        )
        .abs()
        .max()
        .item()
    )

    raw_layers_1_to_25 = torch.stack([
        state[
            0,
            prompt_length - 1:
            prompt_length + generated_length - 1,
            :,
        ]
        for state in outputs.hidden_states[1:-1]
    ])
    if raw_layers_1_to_25.shape[0] != 25:
        raise RuntimeError(
            "Expected raw outputs for layers 1-25."
        )

    raw_layers = torch.cat(
        [
            raw_layers_1_to_25,
            raw_layer26.unsqueeze(0),
        ],
        dim=0,
    )
    expected_shape = (
        NUM_LAYERS,
        generated_length,
        HIDDEN_SIZE,
    )
    if tuple(raw_layers.shape) != expected_shape:
        raise RuntimeError(
            f"Expected {expected_shape}, "
            f"found {tuple(raw_layers.shape)}."
        )

    raw_layers_cpu = (
        raw_layers
        .to(device="cpu", dtype=torch.float16)
        .numpy()
    )

    del outputs
    del raw_layer26
    del post_norm_final
    del reconstructed_post_norm
    del raw_layers_1_to_25
    del raw_layers
    del full_input_ids
    del attention_mask
    del captured

    return (
        raw_layers_cpu,
        replay_seconds,
        {
            "verified": True,
            "rtol": tolerance,
            "atol": tolerance,
            "max_abs_error": max_abs_error,
        },
    )



## In-memory validation, atomic persistence, and lightweight resume

Completed keys are the intersection of JSON and NPZ filename stems. Resume
never opens or hashes completed NPZ files. A partial pair is regenerated.
`records.jsonl` is rebuilt by reading committed JSON records only.



In [8]:
def artifact_paths(sample_id, rollout_id):
    key = f"{sample_id}_r{rollout_id:02d}"
    return (
        key,
        RECORDS_DIR / f"{key}.json",
        HIDDEN_DIR / f"{key}.npz",
    )


def artifact_key_sets():
    record_keys = {
        path.stem
        for path in RECORDS_DIR.glob("*.json")
    }
    hidden_keys = {
        path.stem
        for path in HIDDEN_DIR.glob("*.npz")
    }
    return record_keys, hidden_keys


def completed_artifact_keys():
    record_keys, hidden_keys = artifact_key_sets()
    return record_keys & hidden_keys


def validate_arrays_in_memory(arrays):
    required = {
        "causal_hidden_states",
        "prompt_token_ids",
        "generated_token_ids",
        "layer_indices",
        "layer_numbers",
    }
    if set(arrays) != required:
        raise RuntimeError(
            f"Unexpected NPZ array keys: {sorted(arrays)}"
        )

    hidden = arrays["causal_hidden_states"]
    prompt_ids = arrays["prompt_token_ids"]
    generated_ids = arrays["generated_token_ids"]
    layer_indices = arrays["layer_indices"]
    layer_numbers = arrays["layer_numbers"]

    expected_shape = (
        NUM_LAYERS,
        len(generated_ids),
        HIDDEN_SIZE,
    )
    if hidden.dtype != np.float16:
        raise RuntimeError(
            f"Hidden dtype must be float16, found {hidden.dtype}."
        )
    if hidden.shape != expected_shape:
        raise RuntimeError(
            f"Hidden shape must be {expected_shape}, "
            f"found {hidden.shape}."
        )
    if (
        prompt_ids.ndim != 1
        or prompt_ids.dtype != np.int64
        or len(prompt_ids) == 0
    ):
        raise RuntimeError(
            "prompt_token_ids must be a non-empty int64 vector."
        )
    if (
        generated_ids.ndim != 1
        or generated_ids.dtype != np.int64
        or len(generated_ids) == 0
    ):
        raise RuntimeError(
            "generated_token_ids must be a non-empty int64 vector."
        )
    if not np.array_equal(
        layer_indices,
        np.arange(NUM_LAYERS, dtype=np.int16),
    ):
        raise RuntimeError("layer_indices are not 0 through 25.")
    if not np.array_equal(
        layer_numbers,
        np.arange(1, NUM_LAYERS + 1, dtype=np.int16),
    ):
        raise RuntimeError("layer_numbers are not 1 through 26.")


def atomic_npz(path, arrays):
    validate_arrays_in_memory(arrays)
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp.npz")
    np.savez(tmp, **arrays)
    with open(tmp, "rb+") as handle:
        os.fsync(handle.fileno())
    os.replace(tmp, path)


def collect_one(sample, rollout_id):
    total_started = time.perf_counter()
    key, record_path, hidden_path = artifact_paths(
        sample["sample_id"],
        rollout_id,
    )

    # Remove a stale commit marker before regenerating a JSON-only partial.
    if record_path.exists() and not hidden_path.exists():
        record_path.unlink()

    generation = generate_one(sample, rollout_id)
    selected_box = select_box_candidate(
        generation["generated_text"],
        generation["was_truncated"],
    )
    marker_char_start = (
        int(selected_box["marker_char_start"])
        if selected_box is not None
        else None
    )
    boxed_token_start, alignment_status = (
        align_marker_to_original_tokens(
            generation["generated_text"],
            generation["original_parse_ids"],
            marker_char_start,
        )
    )
    selected_status = (
        str(selected_box["status"])
        if selected_box is not None
        else None
    )
    verified_boundary = (
        alignment_status
        == "verified_original_token_boundary"
        and boxed_token_start is not None
    )
    phase_ranges = (
        split_phases(boxed_token_start)
        if selected_status == "complete"
        and verified_boundary
        else None
    )
    incomplete_or_truncated_phase_ranges = (
        split_phases(boxed_token_start)
        if selected_status in INCOMPLETE_BOX_STATUSES
        and verified_boundary
        else None
    )

    (
        predicted,
        reference,
        extraction_method,
        correct,
    ) = evaluate_answer(
        generation["generated_text"],
        sample["reference_solution"],
        selected_box,
        generation["was_truncated"],
    )
    if (
        selected_status in INCOMPLETE_BOX_STATUSES
        and correct is not None
    ):
        raise AssertionError(
            "Incomplete or truncated answer correctness must be null."
        )
    if (
        generation["was_truncated"]
        and selected_status != "complete"
        and correct is not None
    ):
        raise AssertionError(
            "Unresolved truncated answer correctness must be null."
        )

    (
        hidden_states,
        replay_seconds,
        final_norm_check,
    ) = replay_raw_causal_layers(
        generation["prompt_input_ids"],
        generation["generated_ids"],
    )

    arrays = {
        "causal_hidden_states": hidden_states,
        "prompt_token_ids": (
            generation["prompt_input_ids"]
            .numpy()
            .astype(np.int64, copy=False)
        ),
        "generated_token_ids": (
            generation["generated_ids"]
            .numpy()
            .astype(np.int64, copy=False)
        ),
        "layer_indices": np.arange(
            NUM_LAYERS,
            dtype=np.int16,
        ),
        "layer_numbers": np.arange(
            1,
            NUM_LAYERS + 1,
            dtype=np.int16,
        ),
    }
    validate_arrays_in_memory(arrays)

    save_started = time.perf_counter()
    atomic_npz(hidden_path, arrays)
    save_seconds = time.perf_counter() - save_started

    record = {
        "sample_id": sample["sample_id"],
        "source_index": int(sample["source_index"]),
        "rollout_id": int(rollout_id),
        "seed": generation["seed"],
        "rollout_seed_scheme": ROLLOUT_SEED_SCHEME,
        "question": sample["question"],
        "generated_text": generation["generated_text"],
        "prompt_token_count": int(
            len(generation["prompt_input_ids"])
        ),
        "generated_token_count": int(
            len(generation["generated_ids"])
        ),
        "decoded_non_stop_token_count": int(
            len(generation["original_parse_ids"])
        ),
        "trailing_special_token_count": (
            generation["trailing_special_token_count"]
        ),
        "predicted_answer_normalized": predicted,
        "reference_answer_normalized": reference,
        "answer_extraction_method": extraction_method,
        "correct": correct,
        "selected_box_status": selected_status,
        "selected_box_marker": (
            selected_box.get("marker")
            if selected_box is not None
            else None
        ),
        "selected_box_char_start": marker_char_start,
        "selected_marker_char_end": (
            selected_box.get("marker_char_end")
            if selected_box is not None
            else None
        ),
        "selected_box_char_end": (
            selected_box.get("box_char_end")
            if selected_box is not None
            else None
        ),
        "boxed_token_start": boxed_token_start,
        "alignment_status": alignment_status,
        "alignment_token_source": (
            "retained original generated token IDs after "
            "trailing stop-token removal"
        ),
        "phase_ranges": phase_ranges,
        "incomplete_or_truncated_phase_ranges": (
            incomplete_or_truncated_phase_ranges
        ),
        "stop_reason": generation["stop_reason"],
        "was_truncated": generation["was_truncated"],
        "hit_max_new_tokens": (
            generation["hit_max_new_tokens"]
        ),
        "max_new_tokens": MAX_NEW_TOKENS,
        "final_token_id": generation["final_token_id"],
        "stop_token_ids": STOP_TOKEN_IDS,
        "hidden_state_npz": str(hidden_path),
        "hidden_state_shape": list(hidden_states.shape),
        "hidden_state_dtype": "float16",
        "layer_indices": list(range(NUM_LAYERS)),
        "layer_numbers": list(range(1, NUM_LAYERS + 1)),
        "causal_state_semantics": (
            "row t is the causal state used to predict "
            "generated token t"
        ),
        "causal_slice": "[P - 1 : P + T - 1]",
        "layer_26_stage": (
            "raw decoder block output before final RMSNorm"
        ),
        "raw_layer26_hook_verified": True,
        "final_rmsnorm_reconstruction": final_norm_check,
        "generation_seconds": round(
            generation["generation_seconds"],
            6,
        ),
        "replay_seconds": round(replay_seconds, 6),
        "save_seconds": round(save_seconds, 6),
        "total_seconds": round(
            time.perf_counter() - total_started,
            6,
        ),
    }

    # JSON is written second and is the commit marker for the NPZ.
    atomic_json(record_path, record)

    del arrays
    del hidden_states
    del generation
    return record


def rebuild_records_jsonl():
    # NPZ names only select committed keys; record data comes from JSON only.
    committed_keys = completed_artifact_keys()
    records = []
    for path in sorted(RECORDS_DIR.glob("*.json")):
        if path.stem not in committed_keys:
            continue
        with open(path, "r", encoding="utf-8") as handle:
            records.append(json.load(handle))
    atomic_jsonl(RECORDS_JSONL, records)
    return records


def run_collection():
    total = NUM_PROMPTS * ROLLOUTS_PER_PROMPT
    stats = Counter()
    record_keys, hidden_keys = artifact_key_sets()
    completed_keys = record_keys & hidden_keys
    partial_keys = record_keys ^ hidden_keys

    print(
        f"Resume index: {len(completed_keys)}/{total} completed; "
        f"{len(partial_keys)} partial"
    )
    progress = tqdm(
        total=total,
        desc=f"{RUN_MODE} collection",
    )
    attempt_index = 0

    for sample in samples:
        for rollout_id in range(ROLLOUTS_PER_PROMPT):
            attempt_index += 1
            key, _, _ = artifact_paths(
                sample["sample_id"],
                rollout_id,
            )

            # Filename-only skip: no completed NPZ is opened or inspected.
            if key in completed_keys:
                stats["skipped"] += 1
                progress.update(1)
                continue

            if key in partial_keys:
                stats["regenerated_partial"] += 1

            try:
                collect_one(sample, rollout_id)
                stats["saved"] += 1
                error_path = ERRORS_DIR / f"{key}.json"
                if error_path.exists():
                    error_path.unlink()
            except Exception as exc:
                stats["failed"] += 1
                atomic_json(
                    ERRORS_DIR / f"{key}.json",
                    {
                        "sample_id": sample["sample_id"],
                        "source_index": int(
                            sample["source_index"]
                        ),
                        "rollout_id": int(rollout_id),
                        "seed": rollout_seed(
                            sample["sample_id"],
                            rollout_id,
                        ),
                        "exception_type": type(exc).__name__,
                        "exception_message": str(exc),
                        "traceback": traceback.format_exc(),
                    },
                )
                print(
                    f"Warning: {key} failed: "
                    f"{type(exc).__name__}: {exc}"
                )
            finally:
                gc.collect()
                if attempt_index % 10 == 0 or stats["failed"]:
                    torch.cuda.empty_cache()
                progress.update(1)

    progress.close()
    return dict(stats)



## Run or resume

This cell performs the selected run. When enabled, smoke-only forced
regeneration removes the JSON commit marker first; the NPZ is then
atomically replaced before a new JSON commit marker is written. Neither
smoke nor resume reopens a saved NPZ.



In [9]:
if RUN_MODE == "smoke" and FORCE_SMOKE_REGENERATE:
    removed_commit_markers = 0
    for sample in samples:
        for rollout_id in range(ROLLOUTS_PER_PROMPT):
            _, record_path, _ = artifact_paths(
                sample["sample_id"],
                rollout_id,
            )
            if record_path.exists():
                record_path.unlink()
                removed_commit_markers += 1
    print(
        "Smoke regeneration: removed "
        f"{removed_commit_markers} JSON commit marker(s)."
    )


COLLECTION_STATS = run_collection()
records = rebuild_records_jsonl()

correctness = Counter(
    "unresolved"
    if record["correct"] is None
    else "correct"
    if record["correct"]
    else "incorrect"
    for record in records
)
alignment = Counter(
    record["alignment_status"]
    for record in records
)

print("Collection:", COLLECTION_STATS)
print("Committed records:", len(records))
print("Correctness:", dict(correctness))
print("Alignment:", dict(alignment))
print("records.jsonl:", RECORDS_JSONL)

if RUN_MODE == "smoke":
    if COLLECTION_STATS.get("failed", 0):
        raise RuntimeError(
            "Smoke run failed; inspect the error JSON before a full run."
        )
    if len(records) != 1:
        raise RuntimeError(
            f"Smoke mode expected one committed record, found {len(records)}."
        )
    print(
        "Smoke run completed without reopening the saved NPZ. "
        "Set RUN_MODE = 'full' and ALLOW_FULL_RUN = True "
        "only after reviewing this record."
    )


Resume index: 10000/10000 completed; 0 partial


Collection: {'skipped': 10000}
Committed records: 10000
Correctness: {'correct': 6285, 'incorrect': 3689, 'unresolved': 26}
Alignment: {'verified_original_token_boundary': 9781, 'no_marker': 219}
records.jsonl: /content/drive/MyDrive/gemma3_1b_it_gsm8k_raw26_v3_lean/full/records.jsonl


In [10]:
unresolved_records = [
    record
    for record in records
    if record["correct"] is None
]

print("Extraction methods:")
print(Counter(
    record["answer_extraction_method"]
    for record in unresolved_records
))

print("\nBox/status breakdown:")
print(Counter(
    (
        record["selected_box_status"],
        record["alignment_status"],
        record["stop_reason"],
    )
    for record in unresolved_records
))

for record in unresolved_records:
    print(
        record["sample_id"],
        record["rollout_id"],
        record["answer_extraction_method"],
        record["selected_box_status"],
        record["alignment_status"],
        record["stop_reason"],
    )

Extraction methods:
Counter({'truncated_without_complete_box': 25, 'last_valid_boxed': 1})

Box/status breakdown:
Counter({(None, 'no_marker', 'max_new_tokens'): 25, ('complete', 'verified_original_token_boundary', 'end_of_turn'): 1})
s0094 4 truncated_without_complete_box None no_marker max_new_tokens
s0196 1 truncated_without_complete_box None no_marker max_new_tokens
s0276 0 truncated_without_complete_box None no_marker max_new_tokens
s0294 2 truncated_without_complete_box None no_marker max_new_tokens
s0317 9 truncated_without_complete_box None no_marker max_new_tokens
s0395 7 truncated_without_complete_box None no_marker max_new_tokens
s0477 4 truncated_without_complete_box None no_marker max_new_tokens
s0540 6 truncated_without_complete_box None no_marker max_new_tokens
s0546 4 truncated_without_complete_box None no_marker max_new_tokens
s0546 5 truncated_without_complete_box None no_marker max_new_tokens
s0697 1 truncated_without_complete_box None no_marker max_new_tokens
s0697 

In [11]:
no_marker_records = [
    record
    for record in records
    if record["alignment_status"] == "no_marker"
]

no_marker_correctness = Counter(
    "unresolved"
    if record["correct"] is None
    else "correct"
    if record["correct"]
    else "incorrect"
    for record in no_marker_records
)

print(no_marker_correctness)

Counter({'correct': 121, 'incorrect': 73, 'unresolved': 25})


In [12]:
print(Counter(
    record["answer_extraction_method"]
    for record in no_marker_records
))

Counter({'last_number_fallback': 194, 'truncated_without_complete_box': 25})


In [13]:
no_marker_resolved = [
    record
    for record in records
    if (
        record["alignment_status"] == "no_marker"
        and record["correct"] is not None
    )
]

print(Counter(
    "correct" if record["correct"] else "incorrect"
    for record in no_marker_resolved
))

Counter({'correct': 121, 'incorrect': 73})


In [16]:
correct_records = [
    record
    for record in records
    if record["correct"] is True
]

print("Extraction methods:")
print(Counter(
    record["answer_extraction_method"]
    for record in correct_records
))

print("\nBox/status breakdown:")
print(Counter(
    (
        record["selected_box_status"],
        record["alignment_status"],
        record["stop_reason"],
    )
    for record in correct_records
))

# for record in correct_records:
#     print(
#         record["sample_id"],
#         record["rollout_id"],
#         record["answer_extraction_method"],
#         record["selected_box_status"],
#         record["alignment_status"],
#         record["stop_reason"],
#     )

Extraction methods:
Counter({'last_valid_boxed': 6164, 'last_number_fallback': 121})

Box/status breakdown:
Counter({('complete', 'verified_original_token_boundary', 'end_of_turn'): 6164, (None, 'no_marker', 'end_of_turn'): 121})
